# Attention supervision control (ResNet-50, seed 42)

Retrains the reference ResNet-50 with two mask-derived mechanisms and checks whether raising attention-lesion overlap changes classification:

- **Mask-guided crop** (train only): crop to the lesion bounding box plus a 20 % margin before augmentation.
- **Attention loss**: `L = L_cls + λ · (1 − EP_cam)`, where `EP_cam` is the fraction of the class activation map (CAM, GAP + FC weights, 7 × 7 from `layer4`) inside the mask average-pooled to 7 × 7.

Everything else (split, focal loss, graduated augmentation, sampler, early stopping, schedule) matches the reference run. Validation and test use the plain resize with no crop.

**Scope.** Four configurations, one seed (42). Reported as a control, not as a calibrated comparison.

**Overlap protocol.** EP in this notebook is CAM-based at 7 × 7 with a soft mask. It is a different measurement from the Grad-CAM audit in `notebooks/03_attention_audit.ipynb` (`layer4[-1].conv3`, 224 × 224, binary mask), so the two sets of EP values are not directly comparable. Cell 7 evaluates the unmodified reference model under this same CAM protocol, which is the like-for-like reference for the rows below.

Result (test, seed 42; source `results/attention_supervision/RESUMEN_EXPERIMENTO_GUIADO.json`). Under this protocol the reference model gives EP 0.451 on correct predictions (n = 1761) and 0.544 on errors (n = 230); the three runs with the attention loss invert that order.

| Configuration | Crop | λ | Test BACC | EP (CAM) |
|---|---|---|---|---|
| Reference ResNet-50 (cell 7) | – | – | 0.8135 | 0.462 |
| Crop only | yes | – | 0.7766 | 0.474 |
| Attention loss only | – | 0.5 | 0.7502 | 0.759 |
| Crop + attention loss | yes | 0.2 | 0.7478 | 0.886 |
| Crop + attention loss | yes | 0.5 | 0.7904 | 0.911 |

With the best configuration in the full system (weighted ensemble + TTA, same two baseline backbones): 0.8314 against 0.8486. Every configuration loses balanced accuracy, so the mechanism is not part of the pipeline.

Run order: 1 (only if Drive shows a stale mount) → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10. Cell 10 only reads the JSON files on Drive.

In [ ]:
# 1 · Optional: forced Drive remount. Only if cell 2 cannot see a folder that exists in Drive.

import shutil, os
from google.colab import drive

try:
    drive.flush_and_unmount()
except Exception as e:
    print("nothing to unmount:", e)
if os.path.isdir("/content/drive"):
    shutil.rmtree("/content/drive", ignore_errors=True)
drive.mount("/content/drive", force_remount=True)

target = "/content/drive/MyDrive/ISIC2018_Task3"
print(sorted(os.listdir(target)) if os.path.isdir(target) else f"{target} not found in this account")

In [ ]:
# 2 · Discovery: split, labels, image paths and masks on Drive. No GPU needed.
# Writes only under EXP_DIR; the reference run folder is read only.

import os, glob, json, csv
import numpy as np

DRIVE_ROOT = "/content/drive/MyDrive/ISIC2018_Task3"
CLASSES = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']
MALIGNANT = {'MEL', 'BCC', 'AKIEC'}

COLAB_DIR = os.path.join(DRIVE_ROOT, "colab")                             # reference run, read only
EXP_DIR = os.path.join(DRIVE_ROOT, "colab_gradcam_guided_experimento")    # this experiment
MASKS_DIR = os.path.join(DRIVE_ROOT, "HAM10000_segmentations_lesion_tschandl")
os.makedirs(EXP_DIR, exist_ok=True)


def _mount_drive():
    if not os.path.isdir("/content/drive"):
        from google.colab import drive
        drive.mount('/content/drive')
    assert os.path.isdir(DRIVE_ROOT), f"{DRIVE_ROOT} not found"


def _find_one(filename, root, required=True):
    hits = glob.glob(os.path.join(root, "**", filename), recursive=True)
    if not hits:
        if required:
            raise FileNotFoundError(f"'{filename}' not found under {root}")
        return None
    return hits[0]


def _load_label_map():
    csv_names = ["ISIC2018_Task3_Training_GroundTruth.csv",
                 "ISIC2018_Task3_Validation_GroundTruth.csv",
                 "ISIC2018_Task3_Test_GroundTruth.csv"]
    label_map = {}
    for name in csv_names:
        path = _find_one(name, DRIVE_ROOT, required=False)
        if path is None:
            continue
        with open(path, newline='') as f:
            for row in csv.DictReader(f):
                onehot = {c: float(row[c]) for c in CLASSES}
                label_map[row['image']] = max(onehot, key=onehot.get)
    assert label_map, "no ground-truth CSV found"
    return label_map


def _index_raw_images():
    per_id = {}
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
        for path in glob.glob(os.path.join(DRIVE_ROOT, "**", ext), recursive=True):
            img_id = os.path.splitext(os.path.basename(path))[0]
            if img_id not in per_id:
                per_id[img_id] = path
    return dict(sorted(per_id.items()))


def _build_mask_index(masks_dir):
    idx = {}
    for p in glob.glob(os.path.join(masks_dir, "**", "*.png"), recursive=True):
        base = os.path.splitext(os.path.basename(p))[0]
        for suf in ("_segmentation", "_Segmentation", "_seg", "_mask", "_Mask"):
            if base.endswith(suf):
                base = base[: -len(suf)]
                break
        idx[base] = p
    return idx


_mount_drive()
if not os.path.isdir(COLAB_DIR):
    raise AssertionError(f"{COLAB_DIR} not found. Seen in {DRIVE_ROOT}: {sorted(os.listdir(DRIVE_ROOT))}. "
                         f"If it exists in Drive, run cell 1 and retry.")

SPLIT_PATH = _find_one("split_assignment.json", COLAB_DIR)
with open(SPLIT_PATH) as f:
    split_assignment = json.load(f)
for sp in ('train', 'val', 'test'):
    print(f"  {sp}: {sum(1 for v in split_assignment.values() if v == sp)} ids")

label_map = _load_label_map()
id2path_drive = _index_raw_images()


def records_for(split_name):
    ids = sorted(iid for iid, sp in split_assignment.items() if sp == split_name)
    return [(iid, id2path_drive[iid], label_map[iid]) for iid in ids
            if iid in id2path_drive and iid in label_map]


train_records_drive = records_for('train')
val_records_drive = records_for('val')
test_records_drive = records_for('test')
print(f"  resolved: train={len(train_records_drive)} val={len(val_records_drive)} test={len(test_records_drive)}")
assert train_records_drive and val_records_drive and test_records_drive, "empty split, check DRIVE_ROOT"

assert os.path.isdir(MASKS_DIR), f"{MASKS_DIR} not found"
mask_index_drive = _build_mask_index(MASKS_DIR)
print(f"  {len(mask_index_drive)} masks")
for split_name, recs in (('train', train_records_drive), ('val', val_records_drive),
                         ('test', test_records_drive)):
    hit = sum(1 for iid, _, _ in recs if iid in mask_index_drive)
    print(f"  {split_name:6s} {hit:5d} / {len(recs):5d} with mask ({100 * hit / len(recs):.1f}%)")

In [ ]:
# 3 · Copy images and masks to local disk once. Reading from Drive per epoch is the bottleneck.
# /content is ephemeral: rerun after a disconnect (only missing files are copied).
# DATA_FRACTION < 1 is a smoke-test mode (stratified, fixed seed); results are not comparable.

import os, random, shutil, time
from concurrent.futures import ThreadPoolExecutor, as_completed

DATA_FRACTION = 1.0
CACHE_DIR = "/content/local_cache"
COPY_WORKERS = 12
_SUBSAMPLE_SEED = 12345   # independent of the training seed

os.makedirs(os.path.join(CACHE_DIR, "images"), exist_ok=True)
os.makedirs(os.path.join(CACHE_DIR, "masks"), exist_ok=True)


def stratified_subsample(records, fraction, seed=_SUBSAMPLE_SEED):
    if fraction >= 1.0:
        return list(records)
    rng = random.Random(seed)
    by_class = {}
    for rec in records:
        by_class.setdefault(rec[2], []).append(rec)
    out = []
    for cls, recs in by_class.items():
        n = max(1, round(len(recs) * fraction))
        out.extend(rng.sample(recs, min(n, len(recs))))
    return out


train_records_sub = stratified_subsample(train_records_drive, DATA_FRACTION)
val_records_sub = stratified_subsample(val_records_drive, DATA_FRACTION)
test_records_sub = stratified_subsample(test_records_drive, DATA_FRACTION)
if DATA_FRACTION < 1.0:
    print(f"[warn] DATA_FRACTION={DATA_FRACTION}: smoke test only")


def _copy_one(args):
    iid, src, dst = args
    if os.path.isfile(dst):
        return 'skipped'
    try:
        shutil.copy2(src, dst)
        return 'ok'
    except Exception as e:
        return f'error:{iid}:{e}'


def _copy_all(tasks, label):
    t0 = time.time()
    ok = skipped = errors = 0
    with ThreadPoolExecutor(max_workers=COPY_WORKERS) as ex:
        for fut in as_completed([ex.submit(_copy_one, t) for t in tasks]):
            res = fut.result()
            if res == 'ok': ok += 1
            elif res == 'skipped': skipped += 1
            else: errors += 1; print(f"    [error] {res}")
    print(f"  [{label}] copied={ok} existing={skipped} errors={errors} ({time.time() - t0:.0f}s)")


def cache_images(records, label):
    tasks = [(iid, src, os.path.join(CACHE_DIR, "images", f"{iid}{os.path.splitext(src)[1]}"))
             for iid, src, _ in records]
    _copy_all(tasks, label)
    return {iid: dst for iid, _, dst in tasks}


def cache_masks(records, mask_index_drive, label):
    needed_ids = {iid for iid, _, _ in records if iid in mask_index_drive}
    tasks = [(iid, mask_index_drive[iid],
              os.path.join(CACHE_DIR, "masks", f"{iid}{os.path.splitext(mask_index_drive[iid])[1]}"))
             for iid in needed_ids]
    _copy_all(tasks, label)
    return {iid: dst for iid, _, dst in tasks}


id2local_train = cache_images(train_records_sub, "train")
id2local_val = cache_images(val_records_sub, "val")
id2local_test = cache_images(test_records_sub, "test")
mask_index = cache_masks(train_records_sub + val_records_sub + test_records_sub, mask_index_drive, "masks")

# from here on, local paths only
train_records = [(iid, id2local_train[iid], cls) for iid, _, cls in train_records_sub]
val_records = [(iid, id2local_val[iid], cls) for iid, _, cls in val_records_sub]
test_records = [(iid, id2local_test[iid], cls) for iid, _, cls in test_records_sub]
print(f"  local: {len(train_records)} train / {len(val_records)} val / {len(test_records)} test, "
      f"{len(mask_index)} masks")

In [ ]:
# 4 · Definitions. No heavy compute.
# Transforms, FocalLoss, EarlyStopping and sampler replicate the reference pipeline.
# Hyperparameters live in the CONFIG block of cell 5.

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, WeightedRandomSampler
from PIL import Image
from collections import Counter
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.metrics import balanced_accuracy_score
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    print("[warn] no GPU: training in cell 5 is not practical on CPU")

AUG_LEVEL_NAMES = {0: "none", 1: "light", 2: "medium", 3: "heavy"}
_NORM = [A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), ToTensorV2()]


def build_transforms(img_size):
    aug0 = A.Compose([A.Resize(img_size, img_size), *_NORM])
    aug1 = A.Compose([A.Resize(img_size, img_size),
                      A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
                      A.Rotate(limit=30, p=0.6), *_NORM])
    aug2 = A.Compose([A.Resize(img_size, img_size),
                      A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
                      A.Rotate(limit=90, p=0.7),
                      A.Affine(translate_percent={"x": 0.05, "y": 0.05},
                               scale=(0.9, 1.1), p=0.4), *_NORM])
    aug3 = A.Compose([A.Resize(img_size, img_size),
                      A.Affine(translate_percent={"x": 0.1, "y": 0.1},
                               scale=(0.9, 1.1), rotate=(-20, 20), p=0.5),
                      A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
                      A.Rotate(limit=180, p=0.7),
                      A.GaussNoise(std_range=(0.02, 0.10), p=0.3),
                      A.MotionBlur(blur_limit=3, p=0.2),
                      A.MedianBlur(blur_limit=3, p=0.1), *_NORM])
    return {'aug': [aug0, aug1, aug2, aug3], 'val': aug0}


def crop_to_mask_bbox(img, mask, margin_frac=0.2):
    # mask bbox + proportional margin, clipped to the image; empty mask -> unchanged
    ys, xs = np.where(mask > 0)
    if len(ys) == 0:
        return img, mask
    y0, y1 = ys.min(), ys.max(); x0, x1 = xs.min(), xs.max()
    h, w = mask.shape[:2]
    mh, mw = int((y1 - y0) * margin_frac), int((x1 - x0) * margin_frac)
    y0 = max(0, y0 - mh); y1 = min(h, y1 + mh + 1)
    x0 = max(0, x0 - mw); x1 = min(w, x1 + mw + 1)
    return img[y0:y1, x0:x1], mask[y0:y1, x0:x1]


class GuidedSkinDataset(Dataset):
    """Records (id, path, class). Returns (image, class_idx, mask, has_mask).
    Missing mask -> zero mask and has_mask=False. The crop is applied only when is_train=True."""
    def __init__(self, records, classes, mask_index, is_train, transforms,
                 use_mask_crop=False, crop_margin_frac=0.2,
                 aug_thr_light=2.0, aug_thr_medium=6.0, aug_thr_heavy=15.0,
                 clinical_aug_boost=None):
        self.samples = list(records)
        self.classes = classes
        self.cls2idx = {c: i for i, c in enumerate(classes)}
        self.is_train = is_train
        self.mask_index = mask_index
        self.use_mask_crop = use_mask_crop
        self.crop_margin_frac = crop_margin_frac
        self._aug = transforms['aug']
        self._val = transforms['val']
        clinical_aug_boost = clinical_aug_boost if clinical_aug_boost is not None else {'MEL': 2}

        if is_train:
            counts = Counter(cls for _, _, cls in self.samples)
            max_count = max(counts.values())
            self.class_aug_level = {}
            for cls, n in counts.items():
                r = max_count / n
                if r < aug_thr_light: level = 0
                elif r < aug_thr_medium: level = 1
                elif r < aug_thr_heavy: level = 2
                else: level = 3
                self.class_aug_level[cls] = level
            for cls, min_level in clinical_aug_boost.items():
                if cls in self.class_aug_level:
                    self.class_aug_level[cls] = max(self.class_aug_level[cls], min_level)
        else:
            self.class_aug_level = {c: 0 for c in classes}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        iid, path, cls = self.samples[idx]
        img = np.array(Image.open(path).convert('RGB'))
        mpath = self.mask_index.get(iid)
        has_mask = mpath is not None
        if has_mask:
            mask = (np.array(Image.open(mpath).convert('L')) > 127).astype(np.uint8)
        else:
            mask = np.zeros(img.shape[:2], dtype=np.uint8)

        if self.is_train and self.use_mask_crop and has_mask:
            img, mask = crop_to_mask_bbox(img, mask, margin_frac=self.crop_margin_frac)

        if self.is_train:
            out = self._aug[self.class_aug_level.get(cls, 0)](image=img, mask=mask)
        else:
            out = self._val(image=img, mask=mask)

        img_t, mask_t = out['image'], out['mask']
        mask_t = mask_t.float() if torch.is_tensor(mask_t) else torch.from_numpy(mask_t).float()
        return img_t, self.cls2idx[cls], mask_t, has_mask


class FocalLoss(nn.Module):
    def __init__(self, gamma=1.0, label_smoothing=0.1, reduction='mean'):
        super().__init__()
        self.gamma = gamma; self.label_smoothing = label_smoothing
        self.reduction = reduction

    def forward(self, logits, targets):
        C = logits.size(1)
        with torch.no_grad():
            smooth = torch.full_like(logits, self.label_smoothing / (C - 1))
            smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)
        log_probs = F.log_softmax(logits, dim=1)
        ce_per_sample = -(smooth * log_probs).sum(dim=1)
        with torch.no_grad():
            p_t = log_probs.exp().gather(1, targets.unsqueeze(1)).squeeze(1)
            fw = (1.0 - p_t) ** self.gamma
        loss = fw * ce_per_sample
        return loss.mean() if self.reduction == 'mean' else loss.sum()


class EarlyStopping:
    def __init__(self, patience=8, min_delta=0.001, checkpoint_path=None, bacc_weight=0.8):
        self.patience = patience; self.min_delta = min_delta
        self.checkpoint_path = checkpoint_path; self.bacc_weight = bacc_weight
        self.best_score = -np.inf; self.best_bacc = 0.0; self.best_loss = np.inf
        self.counter = 0; self.best_epoch = 0

    def _score(self, val_bacc, val_loss):
        return self.bacc_weight * val_bacc + (1 - self.bacc_weight) / (1 + val_loss)

    def step(self, val_bacc, val_loss, model, epoch):
        score = self._score(val_bacc, val_loss)
        if score > self.best_score + self.min_delta:
            self.best_score = score; self.best_bacc = val_bacc
            self.best_loss = val_loss; self.counter = 0; self.best_epoch = epoch
            if self.checkpoint_path:
                torch.save(model.state_dict(), self.checkpoint_path)
        else:
            self.counter += 1
        return self.counter >= self.patience

    def state_dict(self):
        return {'best_score': self.best_score, 'best_bacc': self.best_bacc,
                'best_loss': self.best_loss, 'counter': self.counter,
                'best_epoch': self.best_epoch}

    def load_state_dict(self, d):
        self.best_score = d['best_score']; self.best_bacc = d['best_bacc']
        self.best_loss = d['best_loss']; self.counter = d['counter']
        self.best_epoch = d['best_epoch']


def build_model_resnet50(num_classes):
    m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m.to(device)


def build_weighted_sampler(dataset):
    counts = Counter(cls for _, _, cls in dataset.samples)
    weights = {cls: 1.0 / n for cls, n in counts.items()}
    sw = torch.tensor([weights[cls] for _, _, cls in dataset.samples], dtype=torch.float)
    return WeightedRandomSampler(sw, len(sw), replacement=True)


def compute_batch_cam(feat, fc_weight, targets):
    # CAM (Zhou et al., 2016): layer4 maps [B,2048,7,7] weighted by the FC row of each target
    w_y = fc_weight[targets]
    cam = torch.einsum('bchw,bc->bhw', feat, w_y)
    return torch.relu(cam)


def soft_energy_proportion(cam, mask, eps=1e-6):
    # mask average-pooled to the CAM grid; fraction of CAM energy inside it
    mask_soft = F.adaptive_avg_pool2d(mask.unsqueeze(1), cam.shape[-2:]).squeeze(1)
    return (cam * mask_soft).sum(dim=(1, 2)) / (cam.sum(dim=(1, 2)) + eps)


def set_seed(seed):
    # init, sampler order and augmentation; the split is fixed by split_assignment.json
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_one_epoch_guided(model, loader, criterion, optimizer,
                           use_attention_loss, attn_weight, feat_holder):
    # with use_attention_loss=False this is the reference train_one_epoch (attn_loss stays 0)
    model.train()
    run_loss = run_cls = run_attn = cor = total = 0
    pa, la = [], []
    for x, y, mask, has_mask in loader:
        x, y, mask = x.to(device), y.to(device), mask.to(device)
        has_mask = has_mask.to(device)
        optimizer.zero_grad()
        logits = model(x)
        cls_loss = criterion(logits, y)

        if use_attention_loss:
            cam = compute_batch_cam(feat_holder['feat'], model.fc.weight, y)
            ep_soft = soft_energy_proportion(cam, mask)
            if has_mask.sum() > 0:
                attn_loss = (1 - ep_soft[has_mask]).mean()
            else:
                attn_loss = torch.zeros((), device=device)
            loss = cls_loss + attn_weight * attn_loss
        else:
            attn_loss = torch.zeros((), device=device)
            loss = cls_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        run_loss += loss.item() * x.size(0)
        run_cls += cls_loss.item() * x.size(0)
        run_attn += attn_loss.item() * x.size(0)
        p = logits.argmax(1)
        cor += (p == y).sum().item(); total += y.size(0)
        pa.extend(p.cpu().numpy()); la.extend(y.cpu().numpy())
    return (run_loss / total, run_cls / total, run_attn / total,
            cor / total, balanced_accuracy_score(la, pa))


def evaluate_guided(model, loader, criterion):
    # classification only; the mask is ignored
    model.eval()
    run_loss = cor = total = 0; pa, la = [], []
    with torch.no_grad():
        for x, y, mask, has_mask in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x); loss = criterion(logits, y)
            run_loss += loss.item() * x.size(0)
            p = logits.argmax(1); cor += (p == y).sum().item(); total += y.size(0)
            pa.extend(p.cpu().numpy()); la.extend(y.cpu().numpy())
    return run_loss / total, cor / total, balanced_accuracy_score(la, pa)


def evaluate_cam_overlap(model, loader, classes):
    """Test BACC (argmax) plus CAM-based EP on the predicted class, for images with a mask.
    Returns (bacc, ep, correct, true_cls) with one entry per masked image."""
    feats = {}
    handle = model.layer4.register_forward_hook(lambda m, i, o: feats.__setitem__('feat', o))
    model.eval()
    pa, la, all_ep, all_correct, all_cls = [], [], [], [], []
    try:
        with torch.no_grad():
            for x, y, mask, has_mask in loader:
                x, y, mask = x.to(device), y.to(device), mask.to(device)
                logits = model(x)
                p = logits.argmax(1)
                pa.extend(p.cpu().numpy()); la.extend(y.cpu().numpy())
                cam = compute_batch_cam(feats['feat'], model.fc.weight, p)
                ep = soft_energy_proportion(cam, mask)
                for i in range(x.size(0)):
                    if bool(has_mask[i]):
                        all_ep.append(float(ep[i]))
                        all_correct.append(bool(p[i] == y[i]))
                        all_cls.append(classes[int(y[i])])
    finally:
        handle.remove()
    return balanced_accuracy_score(la, pa), np.array(all_ep), all_correct, all_cls


def summarise_overlap(ep, correct, true_cls, classes, malignant):
    def _m(v):
        return float(np.mean(v)) if len(v) else None
    cor = [e for e, c in zip(ep, correct) if c]
    inc = [e for e, c in zip(ep, correct) if not c]
    return {
        'n_masked': int(len(ep)),
        'ep_global': _m(ep),
        'ep_per_class': {c: _m([e for e, k in zip(ep, true_cls) if k == c]) for c in classes
                         if any(k == c for k in true_cls)},
        'ep_malignant': _m([e for e, k in zip(ep, true_cls) if k in malignant]),
        'ep_benign': _m([e for e, k in zip(ep, true_cls) if k not in malignant]),
        'ep_correct': _m(cor), 'n_correct': len(cor),
        'ep_incorrect': _m(inc), 'n_incorrect': len(inc),
        'ep_protocol': 'cam_layer4_7x7_softmask_predclass',
    }


print(f"[setup] definitions loaded, device: {device}")

In [ ]:
# 5 · Training with per-epoch checkpoint on Drive. GPU required.
# Defaults = reference ResNet-50 run; the only change is the mask mechanism selected below.
# Resuming reuses the stored state: if a hyperparameter changes, set RESUME=False.

import os, json, time
import numpy as np
import torch
from torch.utils.data import DataLoader

# ---- CONFIG ----------------------------------------------------------------
SEED = 42                     # only seed 42 was run
RESUME = True

USE_MASK_CROP = True          # crop to lesion bbox (train only)
USE_ATTENTION_LOSS = True     # CAM vs mask loss
ATTENTION_LOSS_WEIGHT = 0.5   # lambda
CROP_MARGIN_FRAC = 0.2

IMG_SIZE = 224
BATCH_SIZE = 32
LR = 1e-4
WEIGHT_DECAY = 1e-4
FOCAL_GAMMA = 1.0
LABEL_SMOOTHING = 0.1

MAX_EPOCHS = 35
ES_PATIENCE = 10
ES_MIN_DELTA = 0.001
ES_BACC_WEIGHT = 0.85
SCHED_PATIENCE = 4
SCHED_FACTOR = 0.5

AUG_THR_LIGHT = 2.0
AUG_THR_MEDIUM = 6.0
AUG_THR_HEAVY = 15.0
CLINICAL_AUG_BOOST = {'MEL': 2}

NUM_WORKERS_TRAIN = 4
NUM_WORKERS_VAL = 2
# ----------------------------------------------------------------------------

set_seed(SEED)

RUN_NAME = (f"crop{int(USE_MASK_CROP)}_attn{int(USE_ATTENTION_LOSS)}_"
            f"w{ATTENTION_LOSS_WEIGHT if USE_ATTENTION_LOSS else 0}_seed{SEED}")
RUN_DIR = os.path.join(EXP_DIR, RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)
STATE_CKPT = os.path.join(RUN_DIR, "training_state_CHECKPOINT.pth")
BEST_CKPT = os.path.join(RUN_DIR, "resnet50_best.pth")
HIST_PATH = os.path.join(RUN_DIR, "history.json")
META_PATH = os.path.join(RUN_DIR, "meta.json")
print(f"[config] {RUN_NAME} -> {RUN_DIR}")

transforms = build_transforms(IMG_SIZE)
train_ds = GuidedSkinDataset(train_records, CLASSES, mask_index, is_train=True,
                             transforms=transforms, use_mask_crop=USE_MASK_CROP,
                             crop_margin_frac=CROP_MARGIN_FRAC,
                             aug_thr_light=AUG_THR_LIGHT, aug_thr_medium=AUG_THR_MEDIUM,
                             aug_thr_heavy=AUG_THR_HEAVY,
                             clinical_aug_boost=CLINICAL_AUG_BOOST)
val_ds = GuidedSkinDataset(val_records, CLASSES, mask_index, is_train=False,
                           transforms=transforms, use_mask_crop=False)
for cls in CLASSES:
    if cls in train_ds.class_aug_level:
        lvl = train_ds.class_aug_level[cls]
        print(f"  {cls:6s} aug level {lvl} ({AUG_LEVEL_NAMES[lvl]})")

sampler = build_weighted_sampler(train_ds)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS_TRAIN, pin_memory=True,
                          persistent_workers=(NUM_WORKERS_TRAIN > 0))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS_VAL, pin_memory=True,
                        persistent_workers=(NUM_WORKERS_VAL > 0))

model = build_model_resnet50(len(CLASSES))
feat_holder = {}
def _capture_feat(mod, inp, out):
    feat_holder['feat'] = out
_hook_handle = model.layer4.register_forward_hook(_capture_feat)

train_crit = FocalLoss(gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)
eval_crit = torch.nn.CrossEntropyLoss()   # no smoothing at validation
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=SCHED_FACTOR, patience=SCHED_PATIENCE)
es = EarlyStopping(patience=ES_PATIENCE, min_delta=ES_MIN_DELTA,
                   checkpoint_path=BEST_CKPT, bacc_weight=ES_BACC_WEIGHT)

history = {k: [] for k in ('train_loss', 'train_cls_loss', 'train_attn_loss',
                           'train_acc', 'train_bacc', 'val_loss', 'val_acc', 'val_bacc')}
start_epoch = 1

if RESUME and os.path.isfile(STATE_CKPT):
    ckpt = torch.load(STATE_CKPT, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    es.load_state_dict(ckpt['es_state'])
    history = ckpt['history']
    start_epoch = ckpt['epoch'] + 1
    print(f"[resume] from epoch {start_epoch} (best so far: epoch {es.best_epoch}, val BACC {es.best_bacc:.4f})")

FULL_CONFIG = {
    'seed': SEED, 'use_mask_crop': USE_MASK_CROP, 'use_attention_loss': USE_ATTENTION_LOSS,
    'attention_loss_weight': ATTENTION_LOSS_WEIGHT, 'crop_margin_frac': CROP_MARGIN_FRAC,
    'img_size': IMG_SIZE, 'batch_size': BATCH_SIZE, 'lr': LR, 'weight_decay': WEIGHT_DECAY,
    'focal_gamma': FOCAL_GAMMA, 'label_smoothing': LABEL_SMOOTHING,
    'max_epochs': MAX_EPOCHS, 'es_patience': ES_PATIENCE, 'es_min_delta': ES_MIN_DELTA,
    'es_bacc_weight': ES_BACC_WEIGHT, 'sched_patience': SCHED_PATIENCE,
    'sched_factor': SCHED_FACTOR, 'aug_thr_light': AUG_THR_LIGHT,
    'aug_thr_medium': AUG_THR_MEDIUM, 'aug_thr_heavy': AUG_THR_HEAVY,
    'clinical_aug_boost': CLINICAL_AUG_BOOST,
}


def save_training_state(epoch):
    torch.save({
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'es_state': es.state_dict(),
        'history': history,
        'epoch': epoch,
        'config': FULL_CONFIG,
    }, STATE_CKPT)


if start_epoch > MAX_EPOCHS or es.counter >= es.patience:
    print(f"[info] run already finished, using best epoch {es.best_epoch}")
else:
    t0 = time.time()
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        tr_loss, tr_cls, tr_attn, tr_acc, tr_bacc = train_one_epoch_guided(
            model, train_loader, train_crit, optimizer,
            use_attention_loss=USE_ATTENTION_LOSS,
            attn_weight=ATTENTION_LOSS_WEIGHT, feat_holder=feat_holder)
        va_loss, va_acc, va_bacc = evaluate_guided(model, val_loader, eval_crit)
        scheduler.step(-es._score(va_bacc, va_loss))

        for k, v in zip(history.keys(),
                        [tr_loss, tr_cls, tr_attn, tr_acc, tr_bacc, va_loss, va_acc, va_bacc]):
            history[k].append(v)

        print(f"  Epoch {epoch:3d}/{MAX_EPOCHS} | train {tr_loss:.4f} (cls {tr_cls:.4f}, attn {tr_attn:.4f}) "
              f"BACC {tr_bacc:.4f} | val {va_loss:.4f} BACC {va_bacc:.4f} | {time.time() - t0:.0f}s")

        stop = es.step(va_bacc, va_loss, model, epoch)
        save_training_state(epoch)   # after es.step, so a resume keeps this epoch's ES decision
        if stop:
            print(f"  early stopping, best epoch {es.best_epoch} (val BACC {es.best_bacc:.4f})")
            break

    with open(HIST_PATH, 'w') as f:
        json.dump(history, f)
    with open(META_PATH, 'w') as f:
        json.dump({'run_name': RUN_NAME, 'val_bacc': es.best_bacc,
                   'val_loss': es.best_loss, 'best_epoch': es.best_epoch,
                   **FULL_CONFIG}, f, indent=2)
    print(f"[ok] {BEST_CKPT}, {HIST_PATH}, {META_PATH}")

assert os.path.isfile(BEST_CKPT), f"{BEST_CKPT} missing"
model.load_state_dict(torch.load(BEST_CKPT, map_location=device, weights_only=False))
_hook_handle.remove()
print(f"[ok] {RUN_NAME}: best weights loaded (epoch {es.best_epoch})")

In [ ]:
# 6 · Test BACC and CAM-based overlap for the run trained in cell 5.

import json
from torch.utils.data import DataLoader

BASELINE_TEST_BACC_RESNET50 = 0.8135   # reference results.json, seed 42

test_ds = GuidedSkinDataset(test_records, CLASSES, mask_index, is_train=False,
                            transforms=transforms, use_mask_crop=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

test_bacc, all_ep, all_correct, all_cls = evaluate_cam_overlap(model, test_loader, CLASSES)
overlap = summarise_overlap(all_ep, all_correct, all_cls, CLASSES, MALIGNANT)
diff = test_bacc - BASELINE_TEST_BACC_RESNET50

print(f"{RUN_NAME}: test BACC {test_bacc:.4f} (reference {BASELINE_TEST_BACC_RESNET50:.4f}, {diff:+.4f})")
print(f"  EP (CAM) overall {overlap['ep_global']:.3f} | malignant {overlap['ep_malignant']:.3f} | "
      f"benign {overlap['ep_benign']:.3f}")
print(f"  correct {overlap['ep_correct']:.3f} (n={overlap['n_correct']}) | "
      f"incorrect {overlap['ep_incorrect']:.3f} (n={overlap['n_incorrect']})")

result_summary = {
    'run_name': RUN_NAME, 'seed': SEED,
    'config': {'use_mask_crop': USE_MASK_CROP, 'use_attention_loss': USE_ATTENTION_LOSS,
               'attention_loss_weight': ATTENTION_LOSS_WEIGHT},
    'test_bacc': float(test_bacc),
    'test_bacc_baseline': BASELINE_TEST_BACC_RESNET50,
    'test_bacc_diff': float(diff),
    **overlap,
}
summary_path = os.path.join(RUN_DIR, 'test_evaluation_summary.json')
with open(summary_path, 'w') as f:
    json.dump(result_summary, f, indent=2)
print(f"[ok] {summary_path}")

In [ ]:
# 7 · Reference ResNet-50 (unmodified, seed 42) under the same CAM protocol as cell 6.
# Needs cells 2-4 only. Inference only; a few minutes on GPU.
# The BACC must reproduce the reference 0.8135; otherwise the checkpoint or preprocessing differs.

import json, glob
import torch.nn as nn
from torch.utils.data import DataLoader

REF_CKPT = sorted(glob.glob(os.path.join(COLAB_DIR, "**", "resnet50_best.pth"), recursive=True))[0]
ref_model = resnet50(weights=None)
ref_model.fc = nn.Linear(ref_model.fc.in_features, len(CLASSES))
_state = torch.load(REF_CKPT, map_location=device, weights_only=False)
ref_model.load_state_dict(_state['state_dict'] if isinstance(_state, dict) and 'state_dict' in _state else _state)
ref_model = ref_model.to(device).eval()

ref_ds = GuidedSkinDataset(test_records, CLASSES, mask_index, is_train=False,
                           transforms=build_transforms(224), use_mask_crop=False)
ref_loader = DataLoader(ref_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

ref_bacc, ref_ep, ref_correct, ref_cls = evaluate_cam_overlap(ref_model, ref_loader, CLASSES)
ref_overlap = summarise_overlap(ref_ep, ref_correct, ref_cls, CLASSES, MALIGNANT)

print(f"reference test BACC {ref_bacc:.4f} (expected 0.8135)")
if abs(ref_bacc - 0.8135) > 0.0005:
    print("[warn] BACC does not match the reference run, do not use the EP values below")
print(f"  EP (CAM) overall {ref_overlap['ep_global']:.3f} | malignant {ref_overlap['ep_malignant']:.3f} | "
      f"benign {ref_overlap['ep_benign']:.3f}")
print(f"  correct {ref_overlap['ep_correct']:.3f} (n={ref_overlap['n_correct']}) | "
      f"incorrect {ref_overlap['ep_incorrect']:.3f} (n={ref_overlap['n_incorrect']})")

ref_path = os.path.join(EXP_DIR, 'reference_cam_protocol.json')
with open(ref_path, 'w') as f:
    json.dump({'checkpoint': REF_CKPT, 'test_bacc': float(ref_bacc), **ref_overlap}, f, indent=2)
print(f"[ok] {ref_path}")
del ref_model

In [ ]:
# 8 · Weighted ensemble without TTA: guided ResNet-50 + the two unmodified reference backbones.
# Weights w_m = val_BACC_m / sum(val_BACC), as in the reference pipeline. Needs model and es from cell 5.

import os, glob, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.models import densenet121, efficientnet_b3
from sklearn.metrics import balanced_accuracy_score
import albumentations as A
from albumentations.pytorch import ToTensorV2

VAL_BACC_DENSENET121_BASELINE = 0.8055       # reference results.json, seed 42
VAL_BACC_EFFICIENTNET_B3_BASELINE = 0.7907
BASELINE_ENSEMBLE_TEST_BACC = 0.8315
IMG_SIZE_DENSENET121 = 224
IMG_SIZE_EFFICIENTNET_B3 = 300


def _load_baseline(m, filename):
    hits = glob.glob(os.path.join(COLAB_DIR, "**", filename), recursive=True)
    assert hits, f"'{filename}' not found under {COLAB_DIR}"
    state = torch.load(hits[0], map_location=device, weights_only=False)
    m.load_state_dict(state['state_dict'] if isinstance(state, dict) and 'state_dict' in state else state)
    print(f"  [ok] {hits[0]}")
    return m.to(device).eval()


def build_densenet121_baseline(num_classes):
    m = densenet121(weights=None)
    m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    return _load_baseline(m, "densenet121_best.pth")


def build_efficientnet_b3_baseline(num_classes):
    m = efficientnet_b3(weights=None)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    return _load_baseline(m, "efficientnet_b3_best.pth")


def _val_transform(img_size):
    return A.Compose([A.Resize(img_size, img_size),
                      A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), ToTensorV2()])


def get_probs(model, records, mask_index, img_size, batch_size=32):
    # softmax on test, in the order of `records`
    ds = GuidedSkinDataset(records, CLASSES, mask_index, is_train=False,
                           transforms={'aug': [_val_transform(img_size)] * 4,
                                       'val': _val_transform(img_size)},
                           use_mask_crop=False)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for x, y, _mask, _has_mask in loader:
            all_probs.append(torch.softmax(model(x.to(device)), dim=1).cpu().numpy())
            all_labels.extend(y.numpy())
    return np.concatenate(all_probs, axis=0), np.array(all_labels)


model_densenet = build_densenet121_baseline(len(CLASSES))
model_efficientnet = build_efficientnet_b3_baseline(len(CLASSES))

probs_resnet, labels_ref = get_probs(model, test_records, mask_index, IMG_SIZE)
probs_densenet, labels_check1 = get_probs(model_densenet, test_records, mask_index, IMG_SIZE_DENSENET121)
probs_efficientnet, labels_check2 = get_probs(model_efficientnet, test_records, mask_index, IMG_SIZE_EFFICIENTNET_B3)
assert np.array_equal(labels_ref, labels_check1) and np.array_equal(labels_ref, labels_check2), "label order mismatch"

bacc_resnet_solo = balanced_accuracy_score(labels_ref, probs_resnet.argmax(1))
bacc_densenet_solo = balanced_accuracy_score(labels_ref, probs_densenet.argmax(1))
bacc_efficientnet_solo = balanced_accuracy_score(labels_ref, probs_efficientnet.argmax(1))
print(f"  ResNet-50 guided {bacc_resnet_solo:.4f} | DenseNet-121 {bacc_densenet_solo:.4f} | "
      f"EfficientNet-B3 {bacc_efficientnet_solo:.4f}")

# JSON keys kept as in the original run files
val_baccs = {
    'resnet50_guiado': es.best_bacc,
    'densenet121': VAL_BACC_DENSENET121_BASELINE,
    'efficientnet_b3': VAL_BACC_EFFICIENTNET_B3_BASELINE,
}
total = sum(val_baccs.values())
weights = {k: v / total for k, v in val_baccs.items()}

ensemble_probs = (weights['resnet50_guiado'] * probs_resnet +
                  weights['densenet121'] * probs_densenet +
                  weights['efficientnet_b3'] * probs_efficientnet)
bacc_ensemble = balanced_accuracy_score(labels_ref, ensemble_probs.argmax(1))
diff = bacc_ensemble - BASELINE_ENSEMBLE_TEST_BACC
print(f"  ensemble {bacc_ensemble:.4f} (reference {BASELINE_ENSEMBLE_TEST_BACC:.4f}, {diff:+.4f})")

out_path = os.path.join(RUN_DIR, "ensemble_ligero_check.json")
with open(out_path, 'w') as f:
    json.dump({
        'individual_test_bacc': {
            'resnet50_guiado': float(bacc_resnet_solo),
            'densenet121_baseline': float(bacc_densenet_solo),
            'efficientnet_b3_baseline': float(bacc_efficientnet_solo),
        },
        'val_bacc_used_for_weights': val_baccs,
        'weights': weights,
        'ensemble_test_bacc': float(bacc_ensemble),
        'ensemble_baseline_test_bacc': BASELINE_ENSEMBLE_TEST_BACC,
        'diff': float(diff),
    }, f, indent=2)
print(f"[ok] {out_path}")

In [ ]:
# 9 · TTA on the same ensemble. Needs cell 8 in the same session.
# 10 rounds, geometric only (RandomRotate90, flips), same as the reference pipeline.
# Batch size per model as in the reference run (EfficientNet-B3 at 300 px uses 8).
# TTA draws are not seeded, as in the reference pipeline.

import os, json, time
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import balanced_accuracy_score
import albumentations as A
from albumentations.pytorch import ToTensorV2

TTA_ROUNDS = 10
BASELINE_TTA_ENSEMBLE_BACC = 0.8486   # reference TTA ensemble, no thresholds


def build_tta_transform(img_size):
    return A.Compose([A.Resize(img_size, img_size),
                      A.RandomRotate90(p=0.5), A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
                      A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), ToTensorV2()])


class TTAImageDataset(Dataset):
    # fresh random transform on every __getitem__, so each pass is a new round
    def __init__(self, records, classes, img_size):
        self.samples = list(records)
        self.cls2idx = {c: i for i, c in enumerate(classes)}
        self.transform = build_tta_transform(img_size)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        iid, path, cls = self.samples[idx]
        out = self.transform(image=np.array(Image.open(path).convert('RGB')))
        return out['image'], self.cls2idx[cls]


def predict_with_tta(model, records, classes, img_size, batch_size, n_rounds=TTA_ROUNDS):
    loader = DataLoader(TTAImageDataset(records, classes, img_size),
                        batch_size=batch_size, shuffle=False, num_workers=2)
    model.eval()
    sum_probs, all_labels = None, []
    t0 = time.time()
    for r in range(1, n_rounds + 1):
        round_probs, round_labels = [], []
        with torch.no_grad():
            for x, y in loader:
                round_probs.append(torch.softmax(model(x.to(device)), dim=1).cpu().numpy())
                round_labels.extend(y.numpy())
        round_probs = np.concatenate(round_probs, axis=0)
        if sum_probs is None:
            sum_probs = round_probs; all_labels = round_labels
        else:
            sum_probs += round_probs
        print(f"    round {r}/{n_rounds} ({time.time() - t0:.0f}s)")
    return sum_probs / n_rounds, np.array(all_labels)


tta_probs_resnet, tta_labels = predict_with_tta(model, test_records, CLASSES, IMG_SIZE, BATCH_SIZE)
tta_probs_densenet, tta_labels_check1 = predict_with_tta(
    model_densenet, test_records, CLASSES, IMG_SIZE_DENSENET121, 32)
tta_probs_efficientnet, tta_labels_check2 = predict_with_tta(
    model_efficientnet, test_records, CLASSES, IMG_SIZE_EFFICIENTNET_B3, 8)
assert np.array_equal(tta_labels, tta_labels_check1) and np.array_equal(tta_labels, tta_labels_check2), \
    "label order mismatch"

tta_bacc_resnet = balanced_accuracy_score(tta_labels, tta_probs_resnet.argmax(1))
tta_bacc_densenet = balanced_accuracy_score(tta_labels, tta_probs_densenet.argmax(1))
tta_bacc_efficientnet = balanced_accuracy_score(tta_labels, tta_probs_efficientnet.argmax(1))
print(f"  TTA individual: ResNet-50 guided {tta_bacc_resnet:.4f} | DenseNet-121 {tta_bacc_densenet:.4f} | "
      f"EfficientNet-B3 {tta_bacc_efficientnet:.4f}")

# same weights as cell 8
tta_ensemble_probs = (weights['resnet50_guiado'] * tta_probs_resnet +
                      weights['densenet121'] * tta_probs_densenet +
                      weights['efficientnet_b3'] * tta_probs_efficientnet)
tta_bacc_ensemble = balanced_accuracy_score(tta_labels, tta_ensemble_probs.argmax(1))
diff = tta_bacc_ensemble - BASELINE_TTA_ENSEMBLE_BACC
print(f"  TTA ensemble {tta_bacc_ensemble:.4f} (reference {BASELINE_TTA_ENSEMBLE_BACC:.4f}, {diff:+.4f})")

out_path = os.path.join(RUN_DIR, "tta_ensemble_check.json")
with open(out_path, 'w') as f:
    json.dump({
        'tta_rounds': TTA_ROUNDS,
        'individual_tta_bacc': {
            'resnet50_guiado': float(tta_bacc_resnet),
            'densenet121_baseline': float(tta_bacc_densenet),
            'efficientnet_b3_baseline': float(tta_bacc_efficientnet),
        },
        'weights': weights,
        'tta_ensemble_bacc': float(tta_bacc_ensemble),
        'tta_ensemble_baseline_bacc': BASELINE_TTA_ENSEMBLE_BACC,
        'diff': float(diff),
    }, f, indent=2)
print(f"[ok] {out_path}")

In [ ]:
# 10 · Consolidation. Reads only the JSON files on Drive (no session state, no GPU).
# Writes RESUMEN_EXPERIMENTO_GUIADO.json / .md in EXP_DIR.

import os, glob, json
from datetime import datetime

DRIVE_ROOT = "/content/drive/MyDrive/ISIC2018_Task3"
EXP_DIR = os.path.join(DRIVE_ROOT, "colab_gradcam_guided_experimento")

REFERENCE = {                       # reference results.json, seed 42
    'resnet50_test_bacc': 0.8135,
    'ensemble_test_bacc': 0.8315,
    'tta_ensemble_test_bacc': 0.8486,
    'clinical_test_bacc': 0.8546,
}

assert os.path.isdir(EXP_DIR), f"{EXP_DIR} not found, mount Drive first"


def _read(path):
    if os.path.isfile(path):
        with open(path) as f:
            return json.load(f)
    return None


ref_cam = _read(os.path.join(EXP_DIR, 'reference_cam_protocol.json'))
if ref_cam is None:
    print("[warn] reference_cam_protocol.json missing (cell 7): no like-for-like EP reference")

runs = []
for rd in sorted(d for d in glob.glob(os.path.join(EXP_DIR, "*")) if os.path.isdir(d)):
    entry = {'run_name': os.path.basename(rd), 'dir': rd}
    for key, fname in (('meta', 'meta.json'), ('test_eval', 'test_evaluation_summary.json'),
                       ('ensemble_ligero', 'ensemble_ligero_check.json'),
                       ('tta_ensemble', 'tta_ensemble_check.json')):
        data = _read(os.path.join(rd, fname))
        if data is not None:
            entry[key] = data
    entry['_has_data'] = any(k in entry for k in ('meta', 'test_eval', 'ensemble_ligero', 'tta_ensemble'))
    runs.append(entry)

valid_runs = [r for r in runs if r['_has_data']]
skipped_runs = [r['run_name'] for r in runs if not r['_has_data']]
if skipped_runs:
    print("[warn] folders without data:", skipped_runs)


def _fmt(v, spec):
    return format(v, spec) if v is not None else '-'


rows = []
for r in valid_runs:
    meta, te = r.get('meta', {}), r.get('test_eval', {})
    attn = meta.get('use_attention_loss')
    test_bacc = te.get('test_bacc')
    rows.append({
        'run': r['run_name'],
        'crop': meta.get('use_mask_crop'),
        'attn': attn,
        'w': meta.get('attention_loss_weight') if attn else None,
        'val_bacc': meta.get('val_bacc'),
        'test_bacc': test_bacc,
        'diff': (test_bacc - REFERENCE['resnet50_test_bacc']) if test_bacc is not None else None,
        'ep_cam': te.get('ep_global'),
    })

header = f"{'run':28s} {'crop':6s} {'attn':6s} {'w':5s} {'val':7s} {'test':7s} {'diff':8s} {'EP_cam':6s}"
print(header); print('-' * len(header))
if ref_cam:
    print(f"{'reference (unmodified)':28s} {'-':6s} {'-':6s} {'-':5s} {'-':7s} "
          f"{ref_cam['test_bacc']:.4f}  {'-':8s} {ref_cam['ep_global']:.3f}")
for x in rows:
    print(f"{x['run']:28s} {str(x['crop']):6s} {str(x['attn']):6s} {_fmt(x['w'], 'g'):5s} "
          f"{_fmt(x['val_bacc'], '.4f'):7s} {_fmt(x['test_bacc'], '.4f'):7s} "
          f"{_fmt(x['diff'], '+.4f'):8s} {_fmt(x['ep_cam'], '.3f')}")

ensemble_rows = [r for r in valid_runs if 'ensemble_ligero' in r or 'tta_ensemble' in r]
for r in ensemble_rows:
    el, tta = r.get('ensemble_ligero'), r.get('tta_ensemble')
    line = f"  {r['run_name']}:"
    if el:
        line += f" ensemble {el['ensemble_test_bacc']:.4f} (ref {REFERENCE['ensemble_test_bacc']:.4f})"
    if tta:
        line += f" | TTA ensemble {tta['tta_ensemble_bacc']:.4f} (ref {REFERENCE['tta_ensemble_test_bacc']:.4f})"
    print(line)

consolidated = {
    'generated_at': datetime.now().isoformat(),
    'reference_baseline': REFERENCE,
    'reference_cam_protocol': ref_cam,
    'ep_protocol_note': ('EP values in this file are CAM-based (layer4, 7x7, soft mask, predicted class). '
                         'They are not comparable with the Grad-CAM audit (layer4[-1].conv3, 224x224, binary mask).'),
    'runs': valid_runs,
    'skipped_runs': skipped_runs,
}
out_json = os.path.join(EXP_DIR, 'RESUMEN_EXPERIMENTO_GUIADO.json')
with open(out_json, 'w') as f:
    json.dump(consolidated, f, indent=2, default=str)

md = [
    "# Attention supervision control (ResNet-50, seed 42)",
    "",
    f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
    "",
    "EP is CAM-based (layer4, 7x7, soft mask, predicted class) and is not comparable with the Grad-CAM audit.",
    "",
    "| Run | Crop | Attention loss | λ | Val BACC | Test BACC | Δ vs reference | EP (CAM) |",
    "|---|---|---|---|---|---|---|---|",
]
if ref_cam:
    md.append(f"| reference (unmodified) | – | – | – | – | {ref_cam['test_bacc']:.4f} | – | {ref_cam['ep_global']:.3f} |")
for x in rows:
    md.append(f"| {x['run']} | {x['crop']} | {x['attn']} | {_fmt(x['w'], 'g')} | "
              f"{_fmt(x['val_bacc'], '.4f')} | {_fmt(x['test_bacc'], '.4f')} | "
              f"{_fmt(x['diff'], '+.4f')} | {_fmt(x['ep_cam'], '.3f')} |")
if ensemble_rows:
    md += ["", "| Run | Ensemble (no TTA) | TTA ensemble | Δ TTA vs reference |", "|---|---|---|---|"]
    for r in ensemble_rows:
        el, tta = r.get('ensemble_ligero'), r.get('tta_ensemble')
        md.append(f"| {r['run_name']} | {_fmt(el and el['ensemble_test_bacc'], '.4f')} | "
                  f"{_fmt(tta and tta['tta_ensemble_bacc'], '.4f')} | {_fmt(tta and tta['diff'], '+.4f')} |")
md += ["", f"Reference: ResNet-50 {REFERENCE['resnet50_test_bacc']:.4f}, ensemble {REFERENCE['ensemble_test_bacc']:.4f}, "
           f"TTA ensemble {REFERENCE['tta_ensemble_test_bacc']:.4f}.", ""]

out_md = os.path.join(EXP_DIR, 'RESUMEN_EXPERIMENTO_GUIADO.md')
with open(out_md, 'w') as f:
    f.write("\n".join(md))
print(f"[ok] {out_json}\n[ok] {out_md}")